# 07 - Per-pixel Phenology & Sub-pixel Unmixing

Two advanced upgrades:
1. **Double-logistic phenology fitting** - data-driven sowing/peak/senescence dates per pixel (replaces fixed Rabi windows).
2. **Linear spectral unmixing (FCLS)** - solves the AWiFS 56 m **mixed-pixel** problem by estimating sub-pixel wheat fractions for smallholder plots.

In [ ]:
import sys
sys.path.append('..')
import numpy as np, matplotlib.pyplot as plt
from src import phenology, unmixing

## 1. Phenology curve fitting
Fit a double-logistic to a wheat NDVI time-series and extract metrics. In production, feed real MODIS/S2 NDVI per pixel (or use `phenology.fit_stack` on an (T,H,W) cube).

In [ ]:
doy = np.arange(0, 180, 8)  # days from 1 Nov
true = phenology.double_logistic(doy, 0.15, 0.85, 35, 140, 0.12, 0.10)
obs = true + np.random.normal(0, 0.03, len(doy))
res = phenology.fit_pixel(doy, obs)
print({k: round(v, 1) for k, v in res.items() if k != 'params'})

fit = phenology.double_logistic(doy, *res['params'])
plt.figure(figsize=(9,4))
plt.scatter(doy, obs, label='observed NDVI', color='gray')
plt.plot(doy, fit, 'g-', lw=2, label='double-logistic fit')
plt.axvline(res['sowing_doy'], color='blue', ls='--', label='sowing')
plt.axvline(res['senescence_doy'], color='red', ls='--', label='senescence')
plt.xlabel('days from 1 Nov'); plt.ylabel('NDVI'); plt.legend(); plt.title('Per-pixel wheat phenology')

In [ ]:
print('NDVI integral (biomass proxy):', round(phenology.ndvi_integral(doy, fit), 2))

## 2. Sub-pixel unmixing for AWiFS 56 m
Three endmembers: wheat, bare soil, other vegetation. FCLS returns abundance fractions per pixel so we can estimate wheat area *within* mixed pixels rather than discarding them.

In [ ]:
# Endmember spectra (Green, Red, NIR, SWIR) - illustrative reflectances
wheat = np.array([0.05, 0.04, 0.45, 0.20])
soil  = np.array([0.15, 0.22, 0.28, 0.35])
other = np.array([0.06, 0.05, 0.38, 0.25])
E = np.vstack([wheat, soil, other])

# Simulate mixed pixels: 60% wheat / 30% soil / 10% other (+ noise)
true_frac = np.array([0.6, 0.3, 0.1])
mix = (true_frac @ E) + np.random.normal(0, 0.005, 4)
ab = unmixing.fcls_unmix(mix, E)
print('Estimated fractions [wheat, soil, other]:', np.round(ab[0], 3))

In [ ]:
# Sub-pixel wheat area over a tile of mixed AWiFS pixels (56 m -> 0.3136 ha)
N = 5000
fracs = np.random.dirichlet([3, 2, 1], size=N)
pixels = fracs @ E + np.random.normal(0, 0.005, (N, 4))
ab = unmixing.fcls_unmix(pixels, E)
area_ha = unmixing.wheat_fraction_area(ab, wheat_idx=0, pixel_area_ha=0.3136)
hard = (ab.argmax(1) == 0).sum() * 0.3136  # hard-classification area
print(f'Sub-pixel wheat area: {area_ha:.1f} ha | hard-classified: {hard:.1f} ha')
print('Hard classification error vs sub-pixel: %.1f%%' % (100*(hard-area_ha)/area_ha))